# Lung Nematic — Modular Colab

Estimate nematic director fields and detect **candidate topological defects**
(`+1/2`, `-1/2`) in lung histology (H&E) images.

Pipeline: tissue mask -> nuclear segmentation -> nuclear-orientation nematic
tensor field (multi-scale) -> winding-number defect detection -> multi-scale
persistence filtering -> overlays, per-nucleus / per-defect CSVs and metrics.

**Recommended:** Runtime -> Change runtime type is not required (CPU is fine).
Run the cells top to bottom.

## 0. Install

In [ ]:
# Installs the package and all dependencies straight from GitHub.
!pip install -q "git+https://github.com/Danpc11/lung-nematic.git"

import lung_nematic
print("lung_nematic version:", lung_nematic.__version__)

## 1. Analysis parameters

`sigmas_px` are the coarse-graining scales (in pixels) for the director field;
a defect must persist across at least `min_scales_for_persistence` of them to be
reported. Set `microns_per_pixel` to your image scale to get defect densities in
mm^-2 (leave 0 if unknown).

In [ ]:
#@title Configure { run: "auto" }
sigmas_px = "40, 55, 70, 85"        #@param {type:"string"}
field_grid_step_px = 64             #@param {type:"integer"}
defect_grid_step_px = 24            #@param {type:"integer"}
min_edge_distance_px = 30           #@param {type:"integer"}
density_quantile = 0.45             #@param {type:"number"}
min_scales_for_persistence = 2      #@param {type:"integer"}
microns_per_pixel = 0.0             #@param {type:"number"}
save_diagnostic_panel = True        #@param {type:"boolean"}

from lung_nematic.config import AnalysisConfig

config = AnalysisConfig(
    sigmas_px=tuple(float(s) for s in sigmas_px.split(",") if s.strip()),
    field_grid_step_px=field_grid_step_px,
    defect_grid_step_px=defect_grid_step_px,
    min_edge_distance_px=min_edge_distance_px,
    density_quantile=density_quantile,
    min_scales_for_persistence=min_scales_for_persistence,
    default_microns_per_pixel=(microns_per_pixel or None),
    save_diagnostic_panel=save_diagnostic_panel,
)
config.validate()
print("Configuration is valid.")
config

## 2. Upload images

Select one or more H&E images (`.jpg`, `.png`, `.tif`, ...). They are copied
into `/content/images`.

In [ ]:
import os, shutil
from google.colab import files

IMAGES_DIR = "/content/images"
os.makedirs(IMAGES_DIR, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join(IMAGES_DIR, name))

print("Images ready:")
for name in sorted(os.listdir(IMAGES_DIR)):
    print("  ", name)

## 3. Run the analysis

Each image takes roughly one minute at full resolution (2560x1920). Results are
written under `/content/results/<image_id>/`.

In [ ]:
from lung_nematic.batch import analyze_folder

summary, errors = analyze_folder(
    input_dir="/content/images",
    output_dir="/content/results",
    config=config,
)

print(f"Processed: {len(summary)} | Failed: {len(errors)}")
if not errors.empty:
    display(errors)
summary

## 4. Inspect overlays and candidate defects

`+` markers are candidate `+1/2` defects, `x` markers candidate `-1/2`.
The director field lines follow local nuclear orientation; line length scales
with the local nematic order `S`.

In [ ]:
import glob
import pandas as pd
from IPython.display import Image as IPyImage, display

overlays = sorted(glob.glob("/content/results/*/*_nematic_overlay.png"))
for overlay in overlays:
    print(os.path.basename(overlay))
    display(IPyImage(overlay, width=900))

defect_files = sorted(glob.glob("/content/results/*/*_defects.csv"))
if defect_files:
    defects = pd.concat(
        [pd.read_csv(f) for f in defect_files], ignore_index=True
    )
    print(f"\nCandidate defects across all images: {len(defects)}")
    display(defects)
else:
    print("No candidate defects were reported for the current settings.")

## 5. Download all results

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/lung_nematic_results", "zip", "/content/results"
)
files.download(archive)